# TF-IDF Vectorizer Tutorial for Billing Codes with Temporal Gaps

This notebook demonstrates how to use scikit-learn's `TfidfVectorizer` with billing code sequences from the MIMIC-IV demo data, including temporal gap tokens.

You will learn how to:
1. Load the MIMIC-IV demo data
2. Insert temporal gap tokens between procedures using `add_gap_entries`
3. Prepare admission-level billing code sequences
4. Fit a TF-IDF vectorizer on code sequences (including gap tokens)
5. Inspect vocabulary and feature importance
6. Transform sequences into TF-IDF feature matrices
7. Save and load the fitted vectorizer

## 1. Imports and Setup

In [5]:
import sys
import pathlib
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))

from icd_gap_utils import add_gap_entries

print("Repo root:", repo_root)

Repo root: c:\Users\user\Documents\daniyal\biling-codes


## 2. Load MIMIC-IV Demo Data

The demo data contains MIMIC-IV procedure ICD codes with columns:
- `subject_id`: Patient identifier
- `hadm_id`: Hospital admission identifier
- `icd_code`: The ICD procedure code
- `chartdate`: Date the code was recorded
- `seq_num`: Sequence position within admission
- `icd_version`: ICD version (will also include "gap" for temporal gap tokens)

In [6]:
data_path = repo_root / "data" / "mimic_4_icd_demo.parquet"
df = pd.read_parquet(data_path)[["subject_id", "hadm_id", "icd_code", "chartdate", "icd_version", "seq_num"]]

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Unique admissions:", df["hadm_id"].nunique())
print("Unique billing codes:", df["icd_code"].nunique())

df.head(10)

Shape: (757313, 6)
Columns: ['subject_id', 'hadm_id', 'icd_code', 'chartdate', 'icd_version', 'seq_num']
Unique admissions: 185412
Unique billing codes: 14490


,subject_id,hadm_id,icd_code,chartdate,icd_version,seq_num
0,14546051,20000069,0KQM0ZZ,2131-11-10,10,1
1,14546051,20000069,10E0XZZ,2131-11-10,10,2
2,13074106,20000102,7359,2135-10-25,9,1
3,13074106,20000102,7309,2135-10-25,9,2
4,14990224,20000147,02100Z9,2121-08-30,10,1
5,14990224,20000147,B211YZZ,2121-08-30,10,2
6,14990224,20000147,021209W,2121-08-30,10,3
7,14990224,20000147,06BQ4ZZ,2121-08-30,10,4
8,14990224,20000147,5A1221Z,2121-08-30,10,5
9,17904239,20000204,6632,2136-01-01,9,1


In [11]:
demo_id = 20000629
df[df.hadm_id == demo_id].sort_values(["chartdate", "seq_num"])

,subject_id,hadm_id,icd_code,chartdate,icd_version,seq_num,icd_code_mapped,pos
38,12726398,20000629,966,2161-12-18,9,2,NaN,NaN
39,12726398,20000629,GAP_1,2161-12-19,gap,2,GAP_1,1.0
40,12726398,20000629,GAP_1,2161-12-20,gap,2,GAP_1,1.0
41,12726398,20000629,GAP_1,2161-12-21,gap,2,GAP_1,1.0
42,12726398,20000629,GAP_1,2161-12-22,gap,2,GAP_1,1.0
43,12726398,20000629,5491,2161-12-23,9,1,NaN,NaN
44,12726398,20000629,GAP_1,2161-12-24,gap,1,GAP_1,1.0
45,12726398,20000629,9904,2161-12-25,9,3,NaN,NaN


## 3. Insert Temporal Gap Tokens

Use `add_gap_entries` to insert synthetic `GAP_<n>` tokens between procedures based on temporal gaps.

In [7]:
# Insert gap tokens for all admissions
df_with_gaps = add_gap_entries(df, gap_list=[30, 10, 5, 1])

n_original = len(df)
n_with_gaps = len(df_with_gaps)
n_gap_rows = n_with_gaps - n_original

print(f"Original rows: {n_original:,}")
print(f"Gap rows added: {n_gap_rows:,}")
print(f"Total rows: {n_with_gaps:,}")
print(f"\nGap token distribution:")
print(df_with_gaps[df_with_gaps['icd_version'] == 'gap']['icd_code'].value_counts())

# Use the data with gaps for the rest of the tutorial
df = df_with_gaps

Original rows: 757,313
Gap rows added: 231,257
Total rows: 988,570

Gap token distribution:
icd_code
GAP_1     199523
GAP_5      22670
GAP_10      8618
GAP_30       446
Name: count, dtype: int64


In [12]:
df[df.hadm_id == demo_id].sort_values(["chartdate", "seq_num"])

,subject_id,hadm_id,icd_code,chartdate,icd_version,seq_num,icd_code_mapped,pos
38,12726398,20000629,966,2161-12-18,9,2,NaN,NaN
39,12726398,20000629,GAP_1,2161-12-19,gap,2,GAP_1,1.0
40,12726398,20000629,GAP_1,2161-12-20,gap,2,GAP_1,1.0
41,12726398,20000629,GAP_1,2161-12-21,gap,2,GAP_1,1.0
42,12726398,20000629,GAP_1,2161-12-22,gap,2,GAP_1,1.0
43,12726398,20000629,5491,2161-12-23,9,1,NaN,NaN
44,12726398,20000629,GAP_1,2161-12-24,gap,1,GAP_1,1.0
45,12726398,20000629,9904,2161-12-25,9,3,NaN,NaN


## 4. Prepare Admission-Level Code Sequences

TF-IDF requires one "document" per sample. Here, each admission is a document containing a sequence of billing codes and gap tokens.

In [13]:
# Sort by admission and date, then group codes by admission
df_sorted = df.sort_values(["hadm_id", "chartdate", "seq_num"])

# Create sequences: each admission gets a list of billing codes (including gap tokens)
admission_sequences = (
    df_sorted
    .groupby("hadm_id")["icd_code"]
    .apply(list)
    .reset_index()
)

admission_sequences.columns = ["hadm_id", "code_sequence"]

print("Number of admissions:", len(admission_sequences))
print("\nSample sequences:")
for i in range(min(3, len(admission_sequences))):
    seq = admission_sequences.iloc[i]
    print(f"  Admission {seq['hadm_id']}: {seq['code_sequence'][:8]}... (length: {len(seq['code_sequence'])})")

Number of admissions: 185412

Sample sequences:
  Admission 20000069: ['0KQM0ZZ', '10E0XZZ']... (length: 2)
  Admission 20000102: ['7359', '7309']... (length: 2)
  Admission 20000147: ['02100Z9', 'B211YZZ', '021209W', '06BQ4ZZ', '5A1221Z']... (length: 5)


## 5. Fit TF-IDF Vectorizer

Since our data is already tokenized (lists of codes), we configure `TfidfVectorizer` to:
- Accept pre-tokenized input via identity functions for `preprocessor` and `tokenizer`
- Set `token_pattern=None` to disable regex tokenization
- Set `lowercase=False` to preserve code formatting

In [14]:
# Create TF-IDF vectorizer for pre-tokenized sequences
vectorizer = TfidfVectorizer(
    preprocessor=lambda x: x,  # Identity function - input is already processed
    tokenizer=lambda x: x,      # Identity function - input is already tokenized
    token_pattern=None,         # Disable regex tokenization
    lowercase=False             # Keep original code case
)

# Fit the vectorizer on all admission sequences
code_sequences = admission_sequences["code_sequence"].values
X_tfidf = vectorizer.fit_transform(code_sequences)

print(f"Feature matrix shape: {X_tfidf.shape}")
print(f"Number of admissions: {X_tfidf.shape[0]}")
print(f"Vocabulary size (unique codes): {X_tfidf.shape[1]}")
print(f"Matrix sparsity: {(1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])) * 100:.2f}%")

Feature matrix shape: (185412, 14494)
Number of admissions: 185412
Vocabulary size (unique codes): 14494
Matrix sparsity: 99.97%


## 6. Inspect Vocabulary and IDF Scores

The vocabulary maps each billing code (and gap token) to a feature index. IDF (Inverse Document Frequency) scores indicate how rare/common each code is across admissions.

In [15]:
# Display first 20 vocabulary entries
vocab_items = list(vectorizer.vocabulary_.items())[:20]
print("Sample vocabulary (code -> feature_index):")
for code, idx in vocab_items:
    print(f"  {code}: {idx}")

Sample vocabulary (code -> feature_index):
  0KQM0ZZ: 6701
  10E0XZZ: 10844
  7359: 12724
  7309: 12717
  02100Z9: 506
  B211YZZ: 13772
  021209W: 533
  06BQ4ZZ: 2527
  5A1221Z: 12473
  6632: 12582
  7534: 12738
  3995: 11570
  GAP_1: 14385
  3723: 11410
  8856: 13418
  GAP_5: 14388
  4523: 11957
  7569: 12748
  734: 12722
  8952: 13466


In [16]:
# Create a DataFrame of codes with their IDF scores
idf_df = pd.DataFrame({
    "icd_code": vectorizer.get_feature_names_out(),
    "idf_score": vectorizer.idf_
})

# Sort by IDF score to see rarest codes (highest IDF) first
idf_df_sorted = idf_df.sort_values("idf_score", ascending=False)

print("\nTop 10 rarest codes (highest IDF):")
print(idf_df_sorted.head(10))

print("\nTop 10 most common codes (lowest IDF):")
print(idf_df_sorted.tail(10))

# Show gap token IDF scores
gap_tokens = idf_df[idf_df['icd_code'].str.startswith('GAP_', na=False)]
if len(gap_tokens) > 0:
    print("\nGap token IDF scores:")
    print(gap_tokens.sort_values('idf_score'))


Top 10 rarest codes (highest IDF):
      icd_code  idf_score
8      00160J2  12.437194
11     00163J2  12.437194
0         0009  12.437194
14492  XW24376  12.437194
14491  XW24346  12.437194
9976   0W903ZX  12.437194
9979   0W913ZZ  12.437194
9980   0W920ZZ  12.437194
9981   0W9230Z  12.437194
9982   0W923ZX  12.437194

Top 10 most common codes (lowest IDF):
      icd_code  idf_score
13621      966   4.233753
30        0040   4.228430
11519     3897   4.174893
11712  3E0G76Z   4.155217
13455     8938   4.067225
13418     8856   4.030821
11516     3893   3.801684
782    02HV33Z   3.729215
14388    GAP_5   3.331659
14385    GAP_1   2.182240

Gap token IDF scores:
      icd_code  idf_score
14385    GAP_1   2.182240
14388    GAP_5   3.331659
14386   GAP_10   4.391285
14387   GAP_30   7.255410


## 7. Examine TF-IDF Features for Sample Admissions

Let's inspect the TF-IDF vectors for individual admissions and see which codes have the highest weights.

In [27]:
def show_top_features(admission_idx, top_n=10):
    """Display top N features for a given admission."""
    hadm_id = admission_idx
    codes = admission_sequences[admission_sequences["hadm_id"] == hadm_id]["code_sequence"].values[0]
    admission_idx = admission_sequences[admission_sequences["hadm_id"] == hadm_id].index[0]
    # Get TF-IDF vector for this admission
    tfidf_vector = X_tfidf[admission_idx].toarray().flatten()
    
    # Get top features
    top_indices = np.argsort(tfidf_vector)[-top_n:][::-1]
    feature_names = vectorizer.get_feature_names_out()
    
    print(f"Admission ID: {hadm_id}")
    print(f"Code sequence length: {len(codes)}")
    print(f"Sample codes: {codes[:10]}...")
    print(f"\nTop {top_n} TF-IDF weighted codes:")
    for i, idx in enumerate(top_indices, 1):
        if tfidf_vector[idx] > 0:
            print(f"  {i}. {feature_names[idx]}: {tfidf_vector[idx]:.4f}")
    print()

# Show features for first 3 admissions

show_top_features(demo_id, top_n=5)

Admission ID: 20000629
Code sequence length: 8
Sample codes: ['966', 'GAP_1', 'GAP_1', 'GAP_1', 'GAP_1', '5491', 'GAP_1', '9904']...

Top 5 TF-IDF weighted codes:
  1. GAP_1: 0.7917
  2. 9904: 0.3814
  3. 5491: 0.3651
  4. 966: 0.3072



## 8. Transform New Sequences

Once fitted, the vectorizer can transform new admission sequences into the same feature space.

In [28]:
# Simulate a new admission with codes and gap tokens
new_admission = [["0038893", "GAP_5", "00HU33Z", "GAP_1", "0016070"]]

# Transform to TF-IDF
new_tfidf = vectorizer.transform(new_admission)

print("New admission codes:", new_admission[0])
print(f"Transformed shape: {new_tfidf.shape}")
print(f"Non-zero features: {new_tfidf.nnz}")

# Show which features have non-zero values
feature_names = vectorizer.get_feature_names_out()
new_vector = new_tfidf.toarray().flatten()
print("\nTF-IDF values:")
for idx in new_tfidf.nonzero()[1]:
    print(f"  {feature_names[idx]}: {new_vector[idx]:.4f}")

New admission codes: ['0038893', 'GAP_5', '00HU33Z', 'GAP_1', '0016070']
Transformed shape: (1, 14494)
Non-zero features: 3

TF-IDF values:
  00HU33Z: 0.8671
  GAP_1: 0.2729
  GAP_5: 0.4166


In [33]:
new_tfidf.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 14494))

## 9. Analyze Code Frequency Statistics

Understanding which codes appear most frequently can provide insights into the dataset.

In [29]:
# Count code occurrences across all admissions (including gap tokens)
code_counts = df["icd_code"].value_counts()

print(f"Total code occurrences (with gaps): {len(df)}")
print(f"Unique codes (including gap tokens): {len(code_counts)}")
print(f"\nTop 15 most frequent codes:")
print(code_counts.head(15))

# Document frequency (how many admissions contain each code)
code_doc_freq = (
    df.groupby("hadm_id")["icd_code"]
    .apply(lambda x: x.unique().tolist())
)

# Flatten and count
all_codes_per_adm = [code for codes in code_doc_freq for code in codes]
doc_freq = pd.Series(all_codes_per_adm).value_counts()

print(f"\nTop 15 codes by document frequency (# admissions):")
print(doc_freq.head(15))

# Separate gap token statistics
gap_codes = df[df['icd_version'] == 'gap']['icd_code'].value_counts()
print(f"\nGap token occurrences:")
print(gap_codes)

Total code occurrences (with gaps): 988570
Unique codes (including gap tokens): 14494

Top 15 most frequent codes:
icd_code
GAP_1      199523
GAP_5       22670
02HV33Z     12893
3893        12509
8856         9137
3897         8682
8938         8630
GAP_10       8618
3E0G76Z      7952
0040         7580
966          7385
8952         6721
9671         6631
9604         6459
0BH17EZ      6172
Name: count, dtype: int64

Top 15 codes by document frequency (# admissions):
GAP_1      56845
GAP_5      18009
02HV33Z    12101
3893       11255
8856        8950
8938        8630
3E0G76Z     7903
3897        7749
0040        7345
966         7306
8952        6721
9671        6375
GAP_10      6241
3722        6002
9604        5976
Name: count, dtype: int64

Gap token occurrences:
icd_code
GAP_1     199523
GAP_5      22670
GAP_10      8618
GAP_30       446
Name: count, dtype: int64
